<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 18 · Portfolio Construction and Risk
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the main Chapter 18 examples:
- estimating annualised expected returns and covariances for a small
  universe,
- shrinking those inputs,
- computing GMV and constrained mean–variance portfolios,
- measuring turnover between portfolios,
- computing asset-level risk contributions, and
- building a simple two-factor model for factor risk decomposition.


### Imports
We work with `NumPy` and `pandas` for portfolio calculations.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
# Configure Matplotlib defaults for the book
mpl.style.use("seaborn-v0_8")  # baseline plotting style
mpl.rcParams.update({"font.family": "serif"})
mpl.rcParams.update({"figure.dpi": 300})


### Returns, Benchmark, and Annualised Inputs
Load the end-of-day dataset, compute daily returns, and estimate annualised
means and covariances for `AAPL`, `JPM`, and `TLT` versus `SPY`.


In [ ]:
LOCAL_EOD = Path("..") / "data" / "eod_data.csv"
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"
source = LOCAL_EOD if LOCAL_EOD.exists() else REMOTE_EOD

prices = pd.read_csv(
    source,
    parse_dates=["Date"],
    index_col="Date",
)

universe = ["AAPL", "JPM", "TLT"]
cols = universe + ["SPY"]
sub = prices[cols].dropna(how="any")

rets_all = sub.pct_change().dropna()
rets = rets_all.iloc[-2 * 252 :]

r_bench = rets["SPY"]
mu_daily = rets[universe].mean()
cov_daily = rets[universe].cov()

mu_annual = (1 + mu_daily) ** 252 - 1
cov_annual = cov_daily * 252
mu_annual

### Shrinking Expected Returns and the Covariance Matrix
We first shrink asset-level expected returns toward the benchmark mean and
then shrink the sample covariance matrix toward its diagonal so that
downstream optimisation is less sensitive to noise.


In [ ]:
# Annualised benchmark mean return and simple shrinkage of expected returns
mu_bench_annual = (1 + r_bench.mean()) ** 252 - 1
shrink = 0.5
mu_shrunk = shrink * mu_annual + (1 - shrink) * mu_bench_annual
mu_shrunk

In [ ]:
# Shrink the covariance matrix toward its diagonal
lam = 0.2
diag_cov = np.diag(np.diag(cov_annual.values))
cov_shrunk = lam * cov_annual.values + (1 - lam) * diag_cov
cov_shrunk = pd.DataFrame(cov_shrunk, index=universe, columns=universe)
cov_shrunk

### GMV and Mean–Variance Portfolios
Using the shrunk inputs, the next cells derive the global minimum-variance
portfolio and an unconstrained mean–variance portfolio before we add practical
constraints such as long-only and position caps.


In [ ]:
Sigma = cov_shrunk.values  # covariance matrix as a NumPy array
ones = np.ones(len(universe))  # vector of ones
inv_Sigma_ones = np.linalg.solve(Sigma, ones)  # Sigma^{-1} 1
w_gmv = inv_Sigma_ones / (ones @ inv_Sigma_ones)  # normalised GMV weights
w_gmv  # inspect GMV weights

In [ ]:
mu_vec = mu_shrunk.values  # shrunk expected returns as array
# unconstrained mean–variance portfolio
w_mv_uncon = np.linalg.solve(Sigma, mu_vec)
w_mv_uncon = w_mv_uncon / w_mv_uncon.sum()  # enforce full investment
w_mv_uncon = pd.Series(w_mv_uncon, index=universe)  # wrap as labelled Series
w_mv_uncon  # inspect unconstrained mean–variance weights

In [ ]:
w_mv_longonly = w_mv_uncon.clip(lower=0)  # enforce non-negative weights
# renormalise to full investment
w_mv_longonly = w_mv_longonly / w_mv_longonly.sum()
cap = 0.35  # per-asset maximum weight
w_mv_capped = w_mv_longonly.clip(upper=cap)  # apply cap
w_mv_capped = w_mv_capped / w_mv_capped.sum()  # renormalise again
w_mv_capped  # inspect constrained weights

### Turnover Between Portfolios
Here we quantify how much trading is required to move from an equal-weight
starting point to the constrained mean–variance portfolio using the standard
half-sum-of-absolute-differences turnover metric.


In [ ]:
def turnover(w_from, w_to):
    w_from = np.asarray(w_from, dtype=float)
    w_to = np.asarray(w_to, dtype=float)
    return 0.5 * np.abs(w_to - w_from).sum()

w_eq = np.repeat(1.0 / len(universe), len(universe))
to_mv = turnover(w_eq, w_mv_capped.values)
to_mv

### Asset-Level Risk Contributions
These cells decompose total portfolio variance into per-asset risk
contributions, which can later be compared against explicit or implicit risk
budgets.


In [ ]:
def risk_contributions(weights, cov):
    w = np.asarray(weights, dtype=float)
    Sigma_rc = np.asarray(cov, dtype=float)
    marginal = Sigma_rc @ w
    total_var = float(w @ marginal)
    contrib = w * marginal
    return contrib, total_var

rc_mv, var_mv = risk_contributions(w_mv_capped.values, Sigma)
rc_mv_pct = pd.Series(rc_mv / var_mv, index=universe)
rc_mv_pct

### Simple Two-Factor Model
We then build a minimal two-factor model with equity (MKT) and rates (RATES)
factors, estimate their covariance, and derive the factor-implied asset
covariance matrix.


In [ ]:
factors = ["MKT", "RATES"]
exposures = pd.DataFrame(
    {"MKT": [1.0, 1.0, 0.0], "RATES": [0.0, 0.0, 1.0]},
    index=universe,
)
exposures

In [ ]:
F = pd.DataFrame(
    {"MKT": rets["SPY"], "RATES": rets["TLT"]},
    index=rets.index,
)
cov_f_annual = F.cov() * 252
cov_f_annual

In [ ]:
B = exposures.values
cov_factor_implied = B @ cov_f_annual.values @ B.T
cov_factor_implied = pd.DataFrame(
    cov_factor_implied,
    index=universe,
    columns=universe,
)
cov_factor_implied

### Factor Risk Contributions
Finally, we decompose portfolio variance by factor, showing how much of the
risk in the constrained portfolio comes from market versus rates exposure.


In [ ]:
w_port = w_mv_capped.values
b_port = exposures.T @ w_port
cov_f = cov_f_annual.values
marginal_f = cov_f @ b_port
var_f = float(b_port @ marginal_f)
rc_f = b_port * marginal_f
rc_f_pct = pd.Series(rc_f / var_f, index=factors)
rc_f_pct

The next cells reproduce the chapter figures for the efficient frontier,
constrained weights, and factor exposures.


In [ ]:
rng = np.random.default_rng(seed=42)
n_samples = 5000
weights_rand = rng.dirichlet(np.ones(len(universe)), size=n_samples)

mu_vec = mu_annual.values
Sigma_front = cov_annual.values

port_rets = weights_rand @ mu_vec
port_vars = np.einsum("ij,jk,ik->i", weights_rand, Sigma_front, weights_rand)
port_vols = np.sqrt(port_vars)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(port_vols, port_rets, s=4, alpha=0.25, label="Random portfolios")
ax.scatter(np.sqrt(w_eq @ Sigma_front @ w_eq), float(w_eq @ mu_vec),
           color="tab:blue", s=60, marker="o", label="Equal-weight")
ax.scatter(np.sqrt(w_gmv @ Sigma_front @ w_gmv), float(w_gmv @ mu_vec),
           color="tab:red", s=60, marker="D", label="GMV")
ax.set_xlabel("Annualised volatility")
ax.set_ylabel("Annualised expected return")
ax.set_title("Sample efficient frontier: equal-weight vs GMV")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()

In [ ]:
weights = pd.concat(
    [w_mv_uncon.rename("Unconstrained"),
     w_mv_longonly.rename("Long-only"),
     w_mv_capped.rename("Long-only, capped")],
    axis=1,
)

x = np.arange(len(universe))
width = 0.25
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width, weights["Unconstrained"], width, label="Unconstrained")
ax.bar(x, weights["Long-only"], width, label="Long-only")
ax.bar(
    x + width,
    weights["Long-only, capped"],
    width,
    label="Long-only, capped",
)
ax.set_xticks(x)
ax.set_xticklabels(universe)
ax.set_ylabel("Portfolio weight")
ax.set_title("Unconstrained vs constrained mean–variance weights")
ax.legend(loc="best")
ax.grid(True, axis="y", linestyle="--", alpha=0.3)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(universe))
width = 0.35
ax.bar(x - width / 2, exposures["MKT"], width, label="MKT")
ax.bar(x + width / 2, exposures["RATES"], width, label="RATES")
ax.set_xticks(x)
ax.set_xticklabels(universe)
ax.set_ylabel("Factor exposure")
ax.set_title("Simple market and rates factor exposures")
ax.set_ylim(0, 1.2)
ax.legend(loc="best")
ax.grid(True, axis="y", linestyle="--", alpha=0.3)
fig.tight_layout()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
